[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch15-responsible-engr/01-danh-gia-tach-theo-nhom.ipynb)

# Đánh giá tách theo nhóm: số trung bình che gì, và cần bao nhiêu mẫu

Chương 15 hỏi vì sao một hệ thống làm đúng điều được bảo vẫn có thể gây hại. Một câu trả lời là số
trung bình: một mô hình đạt độ chính xác tổng thể cao vẫn có thể hỏng với một nhóm người dùng, và
con số tổng hợp không cho thấy điều đó. Sách gọi đây là khuyết điểm của số trung bình (flaw of
averages), và lời đáp của nó là đánh giá tách theo nhóm: tính chỉ số riêng cho từng nhóm con.

Nhưng tách theo nhóm lại gặp một bài toán thống kê. Nhóm càng nhỏ, tập kiểm tra càng ít mẫu của
nhóm, và con số đo được càng ít đáng tin. Sổ tay này đi qua cả hai nửa.

**Bạn sẽ làm:**
1. tính lại bảng 2 của Gender Shades, rồi xem một con số tổng hợp, và cả hai cách phân tích theo
   từng yếu tố, che mất nhóm chịu sai số lớn nhất;
2. lặp lại phép tính nhẩm 1.1: cần bao nhiêu ảnh để kiểm chứng một nhóm chiếm 1 phần trăm người dùng;
3. mô phỏng hàng nghìn tập kiểm tra 10,000 ảnh, và đo xem khoảng tin cậy quen thuộc sai ở đâu khi
   nhóm chỉ có 100 mẫu;
4. vẽ cỡ tập kiểm tra cần có theo tỉ lệ của nhóm, khi lấy mẫu ngẫu nhiên và khi lấy mẫu phân tầng;
5. viết một cổng phát hành có ba kết quả: đạt, không đạt, và chưa đủ bằng chứng;
6. đo thật trên bộ chữ số `load_digits`: một nhóm con chỉ chiếm 2 phần trăm dữ liệu, rồi xem cổng
   nói gì, và thu dữ liệu có mục tiêu thì đổi được gì.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
from scipy import stats
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định
XAM = "0.45"

## 1. Điều số trung bình che giấu: bảng 2

Gender Shades, một cuộc kiểm toán năm 2018, đo sai số phân loại giới tính của các hệ thống thương
mại trên bốn nhóm, tách theo màu da và giới. Bảng 2 của chương ghi sai số của Face++ ở từng nhóm.
Mốc so sánh là nhóm nam da sáng.

In [ ]:
LOI = {  # bảng 2: sai số của Face++, phần trăm
    "nam da sáng": sach(0.8, nguon="0.8 · 9.8 · 0.7 · 34.5 phần trăm"),
    "nữ da sáng": sach(9.8, nguon="0.8 · 9.8 · 0.7 · 34.5 phần trăm"),
    "nam da tối": sach(0.7, nguon="0.8 · 9.8 · 0.7 · 34.5 phần trăm"),
    "nữ da tối": sach(34.5, nguon="0.8 · 9.8 · 0.7 · 34.5 phần trăm"),
}
MOC = LOI["nam da sáng"]
for ten, e in LOI.items():
    print(f"{ten:>12}: sai số {e:5.1f} %, độ chính xác {100 - e:5.1f} %, gấp {e / MOC:6.3f} lần mốc")

theo_sach("độ chính xác, nam da sáng", 100 - LOI["nam da sáng"], sach=99.2, nguon="99.2 · 65.5 phần trăm")
theo_sach("độ chính xác, nữ da tối", 100 - LOI["nữ da tối"], sach=65.5, nguon="99.2 · 65.5 phần trăm")
theo_sach("nam da tối so với mốc", LOI["nam da tối"] / MOC, sach=0.9, nguon="12.2 · 0.9 · 43.1 lần")
theo_sach("nữ da tối so với mốc", LOI["nữ da tối"] / MOC, sach=43.1, nguon="12.2 · 0.9 · 43.1 lần")
print(f"nữ da sáng so với mốc: tính được {LOI['nữ da sáng'] / MOC:.2f} · sách in 12.2")

Ba tỉ số khớp sách. Tỉ số thứ tư cần một dòng giải thích: $9.8 / 0.8$ bằng đúng 12.25, và sách in
12.2, tức làm tròn chữ số 5 xuống. Cả hai cách đọc đều đúng với hai con số của bảng, nên sổ tay in cả
hai mà không khẳng định bên nào.

### Dự đoán

Bảng 2 không cho biết mỗi nhóm chiếm bao nhiêu ảnh trong bộ kiểm tra. Giả sử (giả định của sổ tay)
nhóm nữ da tối chỉ chiếm 10 phần trăm ảnh, ba nhóm còn lại chia đều phần còn lại. Sai số tổng hợp sẽ
là bao nhiêu? Con số ấy có gợi ý gì về 34.5 phần trăm không?

In [ ]:
def sai_so_tong_hop(ti_le_nu_da_toi):
    """Sai số tổng hợp khi nhóm nữ da tối chiếm một tỉ lệ, ba nhóm kia chia đều phần còn lại."""
    phan = {k: (1 - ti_le_nu_da_toi) / 3 for k in LOI}
    phan["nữ da tối"] = ti_le_nu_da_toi
    return sum(phan[k] * LOI[k] for k in LOI)


for ti_le in (0.05, 0.10, 0.25):
    print(f"nữ da tối chiếm {ti_le:.0%} ảnh (giả định): sai số tổng hợp {sai_so_tong_hop(ti_le):.2f} %")

# Phân tích theo từng yếu tố, với bốn nhóm bằng nhau (giả định): chỉ theo giới, hoặc chỉ theo màu da.
theo_gioi = {"nam": (LOI["nam da sáng"] + LOI["nam da tối"]) / 2, "nữ": (LOI["nữ da sáng"] + LOI["nữ da tối"]) / 2}
theo_da = {"da sáng": (LOI["nam da sáng"] + LOI["nữ da sáng"]) / 2, "da tối": (LOI["nam da tối"] + LOI["nữ da tối"]) / 2}
print(f"chỉ theo giới: nam {theo_gioi['nam']:.2f} %, nữ {theo_gioi['nữ']:.2f} %")
print(f"chỉ theo màu da: da sáng {theo_da['da sáng']:.2f} %, da tối {theo_da['da tối']:.2f} %")
print(f"nhóm tệ nhất mà hai cách ấy thấy: {max(theo_gioi['nữ'], theo_da['da tối']):.2f} %, "
      f"còn chỗ giao nhau là {LOI['nữ da tối']} %")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9), gridspec_kw={"width_ratios": [1.15, 1]})
ti_le = np.linspace(0, 0.5, 101)
ax1.plot(100 * ti_le, [sai_so_tong_hop(t) for t in ti_le], color=MAU[0], lw=2, label="sai số tổng hợp")
ax1.axhline(LOI["nữ da tối"], color=MAU[1], lw=1.5, ls="--", label="sai số của nhóm nữ da tối")
ax1.axhline(LOI["nam da sáng"], color=MAU[2], lw=1.5, ls=":", label="sai số của nhóm nam da sáng")
ax1.plot([10], [sai_so_tong_hop(0.10)], "o", color=MAU[0], ms=7)
ax1.annotate(f"{sai_so_tong_hop(0.10):.1f} % khi nhóm chiếm 10 %", (10, sai_so_tong_hop(0.10)),
             xytext=(16, 2.2), fontsize=8, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax1.set_xlim(0, 50)
ax1.set_ylim(0, 40)
ax1.set_xlabel("tỉ lệ ảnh của nhóm nữ da tối trong bộ kiểm tra (%, giả định)")
ax1.set_ylabel("sai số (%)")
ax1.set_title("Cùng bốn sai số nhóm, tổng hợp đổi theo thành phần")
ax1.legend(loc="center left", bbox_to_anchor=(0.01, 0.66), fontsize=8)

nhan = ["nam", "nữ", "da sáng", "da tối", "nam da sáng", "nữ da sáng", "nam da tối", "nữ da tối"]
gia_tri = [theo_gioi["nam"], theo_gioi["nữ"], theo_da["da sáng"], theo_da["da tối"]] + list(LOI.values())
mau_cot = [MAU[0]] * 2 + [MAU[3]] * 2 + [MAU[2]] * 4
ax2.barh(range(len(nhan)), gia_tri, color=mau_cot, height=0.6)
for i, v in enumerate(gia_tri):
    ax2.text(v + 0.5, i, f"{v:.2f} %", va="center", fontsize=8)
ax2.set_yticks(range(len(nhan)))
ax2.set_yticklabels(nhan, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlim(0, 42)
ax2.set_xlabel("sai số (%)")
ax2.set_title("Theo giới, theo màu da, và chỗ giao nhau")
for y_nhom, chu in ((0.5, "chỉ theo giới"), (2.5, "chỉ theo màu da"), (5.5, "giao nhau (bảng 2)")):
    ax2.text(41, y_nhom, chu, ha="right", va="center", fontsize=8, color=XAM)
for y_ke in (1.5, 3.5):
    ax2.axhline(y_ke, color=XAM, lw=0.6)
plt.tight_layout()
plt.show()

Bốn sai số nhóm đứng yên, còn con số tổng hợp trượt theo thành phần của bộ kiểm tra. Ở thành phần nào
thì con số ấy cũng không chứa tỉ số 43.1 lần. Phân tích theo từng yếu tố khá hơn một chút, nhưng
cũng hụt: nhóm "nữ" và nhóm "da tối" đều trộn một nhóm sai rất ít với nhóm sai nhiều nhất. Đó là lý do
chương đòi phân tích giao thoa, tức xét tổ hợp các thuộc tính, vì tác hại có thể dồn vào đúng chỗ giao
nhau.

Hai ví dụ khác của chương cùng một dạng. Một mô hình đạt 95 phần trăm độ chính xác tổng thể nhưng chỉ
78 phần trăm với một lát cắt, như người thu nhập thấp. Và một mô hình cho vay duyệt 95 phần trăm người
đủ điều kiện của nhóm đa số nhưng từ chối 40 phần trăm người đủ điều kiện ngang bằng của nhóm thiểu
số. Lát cắt lớn tới đâu thì hai con số đầu còn đứng chung được?

In [ ]:
TONG = sach(95, nguon="95 · 78 phần trăm")
LAT = sach(78, nguon="95 · 78 phần trăm")
for phan in (0.05, 0.10, 0.20):
    con_lai = (TONG - LAT * phan) / (1 - phan)
    print(f"lát cắt chiếm {phan:.0%}: phần còn lại phải đạt {con_lai:.2f} %")
phan_toi_da = (100 - TONG) / (100 - LAT)
print(f"lát cắt lớn nhất mà vẫn khớp hai con số: {phan_toi_da:.1%} (phần còn lại khi ấy đạt đúng 100 %)")

DUYET_DA_SO = sach(95, nguon="95 phần trăm · 40 phần trăm")
TU_CHOI_THIEU_SO = sach(40, nguon="95 phần trăm · 40 phần trăm")
for phan in (0.05, 0.10, 0.20):
    tong_hop = (1 - phan) * DUYET_DA_SO + phan * (100 - TU_CHOI_THIEU_SO)
    print(f"nhóm thiểu số chiếm {phan:.0%} người đủ điều kiện (giả định): "
          f"tỉ lệ duyệt tổng hợp {tong_hop:.1f} %, nhóm thiểu số {100 - TU_CHOI_THIEU_SO} %")

Lát cắt 78 phần trăm chỉ có thể chiếm tối đa khoảng một phần năm dữ liệu, nếu không thì phần còn lại
phải vượt 100 phần trăm. Với một lát cắt nhỏ, con số tổng hợp gần như là con số của nhóm lớn. Ở ví dụ
cho vay cũng vậy: khi nhóm thiểu số nhỏ, tỉ lệ duyệt tổng hợp vẫn cao và mất mát tổng thể vẫn thấp,
đúng như chương nói về khoảng cách trách nhiệm.

## 2. Thống kê của tính đại diện: phép tính nhẩm 1.1

Muốn đo sai số của một nhóm thì phải có mẫu của nhóm ấy. Phép tính nhẩm thứ nhất của chương: một đội
cần kiểm chứng rằng mô hình Face ID chạy tốt cho một nhóm thiểu số chiếm 1 phần trăm người dùng.

Một tỉ lệ đo trên $n$ mẫu có sai số biên, ở độ tin cậy 95 phần trăm, xấp xỉ
$z\sqrt{p(1-p)/n}$ với $z \approx 1.96$. Tích $p(1-p)$ lớn nhất khi $p = 0.5$, nên đặt $p = 0.5$ cho
ra sai số biên xấu nhất, đúng với mọi tỉ lệ lỗi. Muốn sai số biên ấy bằng $E$ thì cần
$n = z^2 \cdot 0.25 / E^2$.

### Dự đoán

Với $E$ bằng 1 điểm phần trăm, $n$ là bao nhiêu? Và nếu đội lấy mẫu ngẫu nhiên từ toàn bộ người dùng,
họ phải thu và gán nhãn tổng cộng bao nhiêu ảnh?

In [ ]:
TI_LE_NHOM = sach(0.01, nguon="1 phần trăm · 1 điểm phần trăm · 95 phần trăm · 10,000 · 0.01")
BIEN = sach(1, nguon="1 phần trăm · 1 điểm phần trăm · 95 phần trăm") / 100  # 1 điểm phần trăm
TIN_CAY = sach(95, nguon="1 điểm phần trăm · 95 phần trăm") / 100
Z = stats.norm.ppf(0.5 + TIN_CAY / 2)

n_can = Z ** 2 * 0.25 / BIEN ** 2
print(f"z ở độ tin cậy {TIN_CAY:.0%}: {Z:.4f}")
print(f"cỡ mẫu cho sai số biên xấu nhất {100 * BIEN:.0f} điểm phần trăm: {n_can:,.1f}, làm tròn lên {math.ceil(n_can):,}")
theo_sach("số ảnh cần cho nhóm", n_can, sach=10_000, nguon="95 phần trăm · 10,000 · 0.01", tol=500)

N_NHOM = sach(10_000, nguon="95 phần trăm · 10,000 · 0.01 · 1,000,000")
tong_ngau_nhien = N_NHOM / TI_LE_NHOM
theo_sach("tổng số ảnh khi lấy mẫu ngẫu nhiên", tong_ngau_nhien, sach=1_000_000, nguon="0.01 · 1,000,000 · 100 lần")
theo_sach("hệ số dữ liệu so với nhóm đa số", tong_ngau_nhien / N_NHOM, sach=100, nguon="0.01 · 1,000,000 · 100 lần")
print(f"lấy mẫu phân tầng, nhắm thẳng vào nhóm: {N_NHOM:,} ảnh")

Công thức cho ra 9,604, và sách viết "khoảng 10,000", tức làm tròn lên. Lấy ngẫu nhiên thì 10,000 ảnh
của nhóm đi kèm 990,000 ảnh của những người khác, nên tổng là 1,000,000, gấp 100 lần.

Chương còn nói chiều ngược lại: một tập kiểm tra 10,000 mẫu, đủ cho nhóm đa số, chỉ chứa khoảng 100
mẫu của nhóm 1 phần trăm. Với 100 mẫu, sai số biên xấu nhất là bao nhiêu?

In [ ]:
TAP_KIEM_TRA = sach(10_000, nguon="10,000 · 100 · 1 phần trăm · 100 lần")
mau_nhom = TAP_KIEM_TRA * TI_LE_NHOM
theo_sach("số mẫu của nhóm 1 phần trăm", mau_nhom, sach=100, nguon="10,000 · 100 · 1 phần trăm")


def sai_so_bien(n, p=0.5):
    """Nửa độ rộng của khoảng tin cậy 95 phần trăm theo xấp xỉ chuẩn."""
    return Z * math.sqrt(p * (1 - p) / n)


for n in (100, 1_000, 10_000):
    print(f"n = {n:>6,}: sai số biên xấu nhất ±{100 * sai_so_bien(n):.1f} điểm phần trăm")

## 3. Mô phỏng: một nhóm 1 phần trăm trong 10,000 mẫu

Giả định của sổ tay: mô hình sai 2 phần trăm với nhóm đa số và 5 phần trăm với nhóm thiểu số, tức nhóm
thiểu số chịu sai số gấp 2.5 lần. Mỗi lần mô phỏng rút ngẫu nhiên một tập kiểm tra 10,000 mẫu từ
toàn bộ người dùng, đếm số mẫu của nhóm, rồi đo sai số của từng nhóm.

### Dự đoán

Qua 4,000 tập kiểm tra như thế, sai số đo được của nhóm thiểu số dao động trong khoảng nào? Có tập nào
cho thấy nhóm thiểu số sai **ít hơn** nhóm đa số không?

In [ ]:
LOI_DA_SO, LOI_THIEU_SO = 0.02, 0.05  # giả định của sổ tay
SO_LAN = 4_000
rng = np.random.default_rng(SEED)
n_thieu_so = rng.binomial(TAP_KIEM_TRA, TI_LE_NHOM, SO_LAN)
n_da_so = TAP_KIEM_TRA - n_thieu_so
do_thieu_so = rng.binomial(n_thieu_so, LOI_THIEU_SO) / n_thieu_so
do_da_so = rng.binomial(n_da_so, LOI_DA_SO) / n_da_so

thap, cao = np.percentile(n_thieu_so, [2.5, 97.5])
print(f"số mẫu của nhóm thiểu số trong mỗi tập: trung bình {n_thieu_so.mean():.1f}, 95 % số tập nằm trong "
      f"[{thap:.0f}, {cao:.0f}]")
for ten, do in (("đa số", do_da_so), ("thiểu số", do_thieu_so)):
    a, b = np.percentile(do, [2.5, 97.5])
    print(f"sai số đo được, nhóm {ten}: 95 % số tập nằm trong [{100 * a:.2f} %, {100 * b:.2f} %]")
print(f"số tập mà nhóm thiểu số đo ra sai ít hơn nhóm đa số: {(do_thieu_so < do_da_so).sum()} / {SO_LAN}")
print(f"số tập đo ra nhóm thiểu số sai 0 lần: {(do_thieu_so == 0).sum()} / {SO_LAN}")
print(f"sai số biên theo công thức ở p = {LOI_THIEU_SO}, n = 100: ±{100 * sai_so_bien(100, LOI_THIEU_SO):.2f} điểm")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
khung = np.arange(0, 0.16, 0.005)
ax.hist(do_da_so, bins=khung, color=MAU[0], alpha=0.75, label="nhóm đa số (khoảng 9,900 mẫu)")
ax.hist(do_thieu_so, bins=khung, color=MAU[1], alpha=0.65, label="nhóm thiểu số (khoảng 100 mẫu)")
ax.axvline(LOI_DA_SO, color=MAU[0], lw=1.5, ls="--")
ax.axvline(LOI_THIEU_SO, color=MAU[1], lw=1.5, ls="--")
ax.text(LOI_DA_SO + 0.006, ax.get_ylim()[1] * 0.92, "sai số thật 2 %", fontsize=8, color=MAU[0])
ax.text(LOI_THIEU_SO, ax.get_ylim()[1] * 0.80, " sai số thật 5 %", fontsize=8, color=MAU[1])
ax.set_xticks(khung[::4])
ax.set_xticklabels([f"{100 * v:.0f}" for v in khung[::4]])
ax.set_xlabel("sai số đo được trên một tập kiểm tra 10,000 mẫu (%)")
ax.set_ylabel("số tập kiểm tra")
ax.set_title(f"{SO_LAN:,} tập kiểm tra rút ngẫu nhiên (giả định: sai 2 % và 5 %)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Nhóm đa số được đo rất sát. Nhóm thiểu số thì không: có tập đo ra gần 0, có tập đo ra gấp đôi sai số
thật. Một đội chỉ có một tập kiểm tra sẽ không biết mình rơi vào đâu trong hình.

Khoảng ấy hẹp hơn sai số biên xấu nhất, vì sai số thật ở đây là 5 phần trăm chứ không phải 50. Nhưng
trước khi đo thì đội chưa biết sai số thật, nên chương lập kế hoạch theo trường hợp xấu nhất.

### Khoảng tin cậy quen thuộc sai ở đâu

Khoảng tin cậy "cộng trừ $z\sqrt{\hat p(1-\hat p)/n}$" (khoảng Wald) dùng chính tỉ lệ đo được. Khi
$n$ nhỏ và tỉ lệ lỗi nhỏ, nó hỏng: một nhóm đo ra 0 lỗi có khoảng Wald độ rộng bằng 0, tức một lời
khẳng định chắc chắn rút ra từ 100 mẫu. Khoảng Wilson sửa điều đó. Ô dưới tính **chính xác** tỉ lệ
phủ của mỗi khoảng ở $n = 100$, tức xác suất khoảng chứa sai số thật, bằng cách cộng xác suất nhị thức
của mọi số lỗi có thể.

**Dự đoán:** với sai số thật 1 phần trăm và 100 mẫu, khoảng Wald "95 phần trăm" chứa sai số thật trong
bao nhiêu phần trăm số lần?

In [ ]:
def wilson(k, n, tin_cay=0.95):
    """Khoảng tin cậy Wilson cho tỉ lệ k/n (k, n là số hoặc mảng NumPy)."""
    z = stats.norm.ppf(0.5 + tin_cay / 2)
    p = np.asarray(k, dtype=float) / n
    mau_so = 1 + z ** 2 / n
    tam = (p + z ** 2 / (2 * n)) / mau_so
    nua = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / mau_so
    return np.clip(tam - nua, 0, 1), np.clip(tam + nua, 0, 1)


def wald(k, n):
    p = np.asarray(k, dtype=float) / n
    nua = Z * np.sqrt(p * (1 - p) / n)
    return p - nua, p + nua


n = 100
k = np.arange(n + 1)
luoi_p = np.linspace(0.002, 0.30, 299)
phu = {"Wald": [], "Wilson": []}
for ten, khoang in (("Wald", wald), ("Wilson", wilson)):
    lo, hi = khoang(k, n)
    for p in luoi_p:
        xac_suat = stats.binom.pmf(k, n, p)
        phu[ten].append(float(xac_suat[(lo <= p) & (p <= hi)].sum()))
for p in (0.01, 0.05, 0.20):
    i = int(np.argmin(abs(luoi_p - p)))
    print(f"sai số thật {p:.0%}, n = {n}: Wald phủ {phu['Wald'][i]:.1%}, Wilson phủ {phu['Wilson'][i]:.1%}")
lo0, hi0 = wilson(0, n)
print(f"0 lỗi trên {n} mẫu: Wald cho [0, 0], Wilson cho [0, {100 * float(hi0):.1f} %]")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(100 * luoi_p, 100 * np.array(phu["Wald"]), color=MAU[1], lw=1.3, label="khoảng Wald")
ax.plot(100 * luoi_p, 100 * np.array(phu["Wilson"]), color=MAU[0], lw=1.3, label="khoảng Wilson")
ax.axhline(95, color=XAM, lw=1, ls="--", label="mức danh nghĩa 95 %")
ax.set_xlim(0, 30)
ax.set_ylim(50, 100)
ax.set_xlabel("sai số thật của nhóm (%)")
ax.set_ylabel("tỉ lệ phủ thật (%)")
ax.set_title("Khoảng tin cậy 95 % trên 100 mẫu: tỉ lệ phủ tính chính xác")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Với tỉ lệ lỗi nhỏ, khoảng Wald phủ thấp hơn hẳn mức nó tự xưng, còn khoảng Wilson bám quanh 95 phần
trăm. Đường răng cưa là thật, không phải nhiễu: số lỗi chỉ nhận giá trị nguyên. Phần sau dùng khoảng
Wilson.

## 4. Cỡ tập kiểm tra theo tỉ lệ của nhóm

Phép tính nhẩm 1.1 là một điểm trên một họ đường cong: với sai số biên $E$ và nhóm chiếm tỉ lệ $s$, lấy
mẫu ngẫu nhiên cần $n_{nhóm}/s$ mẫu, còn lấy mẫu phân tầng chỉ cần $n_{nhóm}$.

In [ ]:
ti_le_s = np.logspace(-3, np.log10(0.5), 200)
fig, ax = plt.subplots(figsize=(10, 4.2))
for bien, mau in ((0.01, MAU[1]), (0.02, MAU[3]), (0.05, MAU[2])):
    n_nhom = math.ceil(Z ** 2 * 0.25 / bien ** 2)
    ax.loglog(100 * ti_le_s, n_nhom / ti_le_s, color=mau, lw=2, label=f"ngẫu nhiên, sai số biên ±{100 * bien:.0f} điểm")
    ax.loglog(100 * ti_le_s, np.full_like(ti_le_s, n_nhom), color=mau, lw=1.2, ls="--")
    ax.text(0.105, n_nhom * 0.86, f"phân tầng: {n_nhom:,}", va="top", fontsize=8, color=mau)
ax.plot([100 * TI_LE_NHOM], [tong_ngau_nhien], "o", color="black", ms=7)
ax.annotate("phép tính nhẩm 1.1:\nnhóm 1 %, 1,000,000 ảnh", (100 * TI_LE_NHOM, tong_ngau_nhien),
            xytext=(2.2, 3e6), fontsize=8, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax.set_xlim(0.1, 50)
ax.set_ylim(150, 2e7)
ax.set_xlabel("tỉ lệ của nhóm trong tổng thể (%)")
ax.set_ylabel("số mẫu phải thu và gán nhãn")
ax.set_title("Kiểm chứng một nhóm: lấy mẫu ngẫu nhiên và lấy mẫu phân tầng")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

Trên trục log, mỗi đường ngẫu nhiên là một đường thẳng dốc xuống: nhóm nhỏ đi 10 lần thì dữ liệu phải
thu tăng 10 lần. Đường phân tầng thì nằm ngang. Khoảng cách giữa hai đường chính là chi phí của việc dựa
vào "phân phối tự nhiên", và theo sách, công bằng đòi hỏi kỹ thuật dữ liệu có chủ đích, không chỉ là
thêm dữ liệu.

## 5. Cổng phát hành có ba kết quả

Chương nói một cổng phát hành phải phân biệt việc thiếu bằng chứng với bằng chứng về sự ngang bằng.
Nếu một lát cắt không đủ mẫu có nhãn để chặn được tỉ lệ lỗi, phải báo là chưa đủ dữ liệu, không phải
là đạt. Không có ngưỡng phổ quát nào, nên sổ tay dùng một chính sách mà chương nêu làm ví dụ: tỉ số tỉ
lệ lỗi giữa nhóm và mốc dưới 1.25 lần.

Cổng dưới so **cả khoảng tin cậy** của tỉ số với ngưỡng, không so con số đo được:

* cả khoảng nằm dưới ngưỡng thì **đạt**;
* cả khoảng nằm trên ngưỡng thì **không đạt**;
* khoảng vắt qua ngưỡng thì **chưa đủ bằng chứng**.

Mỗi nhóm lấy một khoảng Wilson 97.5 phần trăm, để hai khoảng cùng đúng với xác suất ít nhất 95 phần
trăm. Tỉ số thấp nhất có thể là cận dưới của nhóm chia cận trên của mốc, và ngược lại.

In [ ]:
TI_SO_TOI_DA = sach(1.25, nguon="1.25 lần · 5 điểm phần trăm")


def cong_phat_hanh(loi_nhom, n_nhom, loi_moc, n_moc, ti_so_toi_da=TI_SO_TOI_DA):
    """Trả về (kết quả, cận dưới, cận trên) của tỉ số tỉ lệ lỗi nhóm / mốc."""
    if n_nhom == 0 or n_moc == 0:
        return "chưa đủ bằng chứng", math.nan, math.nan
    lo_n, hi_n = (float(v) for v in wilson(loi_nhom, n_nhom, 0.975))
    lo_m, hi_m = (float(v) for v in wilson(loi_moc, n_moc, 0.975))
    duoi = lo_n / hi_m
    tren = hi_n / lo_m if lo_m > 0 else math.inf
    if tren < ti_so_toi_da:
        return "đạt", duoi, tren
    if duoi > ti_so_toi_da:
        return "không đạt", duoi, tren
    return "chưa đủ bằng chứng", duoi, tren


# Một mốc đo rất kỹ: 100,000 mẫu, sai 2 phần trăm (giả định của sổ tay).
N_MOC, LOI_MOC = 100_000, 0.02
ket_qua_cong = {}
for loi_that, mo_ta in ((0.02, "ngang bằng thật"), (0.05, "gấp 2.5 lần")):
    for n_nhom in (100, 1_000, 10_000, 100_000):
        kq, duoi, tren = cong_phat_hanh(round(loi_that * n_nhom), n_nhom, round(LOI_MOC * N_MOC), N_MOC)
        ket_qua_cong[(loi_that, n_nhom)] = (kq, duoi, tren)
        print(f"nhóm {mo_ta:>15}, {n_nhom:>7,} mẫu: tỉ số trong [{duoi:.2f}, {tren:.2f}] → {kq}")

# Tính chất tất định: số liệu ở đây là số lỗi kỳ vọng, nên kết quả không đổi giữa các máy.
assert ket_qua_cong[(0.02, 100)][0] == ket_qua_cong[(0.05, 100)][0] == "chưa đủ bằng chứng"
assert ket_qua_cong[(0.05, 1_000)][0] == "không đạt"
assert ket_qua_cong[(0.02, 100_000)][0] == "đạt"

Ở 100 mẫu, đúng cỡ mà tập kiểm tra 10,000 mẫu dành cho nhóm 1 phần trăm, cổng không phân biệt được
một nhóm ngang bằng thật với một nhóm sai gấp 2.5 lần, nên nó trả lời "chưa đủ bằng chứng" cho cả hai.
Một cổng chỉ nhìn con số đo được sẽ cho nhóm thứ nhất "đạt", và may rủi của việc lấy mẫu ở phần 3 có
thể cho cả nhóm thứ hai "đạt".

Bảng còn cho thấy một điều chương không nói thẳng: chứng minh **có** chênh lệch rẻ hơn chứng minh
**không có**. Chênh lệch gấp 2.5 lần lộ ra ở 1,000 mẫu, còn muốn khẳng định tỉ số dưới 1.25 lần thì cần
cỡ mười nghìn mẫu trở lên, vì khoảng tin cậy phải lọt hẳn vào dưới ngưỡng.

## 6. Đo thật: một nhóm con 2 phần trăm trên `load_digits`

Bảng 5 của chương nêu một rủi ro là lệch bối cảnh triển khai: mô hình học trên dữ liệu thu ở một điều
kiện, rồi chạy trên dữ liệu thu ở điều kiện khác. Sổ tay dựng một nhóm con như thế (giả định): ảnh chữ
số bị lệch sang phải 2 điểm ảnh, như ảnh từ một loại máy quét đặt lệch. Nhóm này chỉ chiếm 2 phần trăm
dữ liệu huấn luyện và 2 phần trăm lưu lượng, đúng như một nhóm ít được thu thập.

Mô hình là hồi quy logistic trên 64 điểm ảnh. Tập kiểm tra tự nhiên có cùng tỉ lệ 2 phần trăm.

### Dự đoán

Độ chính xác tổng thể trên tập kiểm tra tự nhiên sẽ gần độ chính xác của nhóm nào hơn? Tập ấy có bao
nhiêu ảnh của nhóm con, và cổng ở phần 5 sẽ nói gì?

In [ ]:
X, y = load_digits(return_X_y=True)
X = X / 16.0


def lech_phai(anh, buoc=2):
    """Dời mỗi ảnh 8 × 8 sang phải `buoc` cột; cột trống bên trái điền 0."""
    a = anh.reshape(-1, 8, 8)
    b = np.zeros_like(a)
    b[:, :, buoc:] = a[:, :, :-buoc]
    return b.reshape(-1, 64)


rng = np.random.default_rng(SEED)
thu_tu = rng.permutation(len(X))
hl, kt = thu_tu[:1200], thu_tu[1200:]
TI_LE_CON = 0.02  # giả định của sổ tay


def tron(chi_so, ti_le, r):
    """Một tập có `ti_le` ảnh thuộc nhóm con (đã dời), phần còn lại giữ nguyên."""
    Xm = X[chi_so].copy()
    con = r.random(len(chi_so)) < ti_le
    Xm[con] = lech_phai(Xm[con])
    return Xm, y[chi_so], con


X_hl, y_hl, con_hl = tron(hl, TI_LE_CON, np.random.default_rng(SEED + 1))
X_kt, y_kt, con_kt = tron(kt, TI_LE_CON, np.random.default_rng(SEED + 2))
mo_hinh = LogisticRegression(max_iter=3000).fit(X_hl, y_hl)
dung = mo_hinh.predict(X_kt) == y_kt
print(f"huấn luyện: {len(hl)} ảnh, {con_hl.sum()} thuộc nhóm con · kiểm tra tự nhiên: {len(kt)} ảnh, "
      f"{con_kt.sum()} thuộc nhóm con")
do_tren_may("độ chính xác tổng thể, tập tự nhiên", 100 * dung.mean(), "%")
do_tren_may("độ chính xác nhóm chính, tập tự nhiên", 100 * dung[~con_kt].mean(), "%")
do_tren_may(f"độ chính xác nhóm con, chỉ {con_kt.sum()} ảnh", 100 * dung[con_kt].mean(), "%")
kq, duoi, tren = cong_phat_hanh(int((~dung[con_kt]).sum()), int(con_kt.sum()),
                                int((~dung[~con_kt]).sum()), int((~con_kt).sum()))
print(f"cổng trên tập tự nhiên: tỉ số sai số trong [{duoi:.2f}, {tren:.2f}] → {kq}")

Độ chính xác tổng thể gần như trùng với nhóm chính, vì nhóm con chỉ góp vài ảnh. Kết quả của cổng thì
tuỳ số lỗi đếm được trên vài ảnh ấy: nếu nhóm con sai rất nhiều, vài ảnh cũng đủ để nói "không đạt";
nếu nó sai ít hơn, khoảng tin cậy rộng sẽ vắt qua ngưỡng. Điều chắc chắn là con số đo trên nhóm con có
khoảng tin cậy rất rộng.

### Lấy mẫu phân tầng cho tập kiểm tra, rồi thu dữ liệu huấn luyện có mục tiêu

Bây giờ đánh giá theo đúng cách phép tính nhẩm 1.1 đề xuất: dựng một tập kiểm tra riêng cho nhóm con,
bằng cách dời toàn bộ ảnh kiểm tra. Rồi thử hai cách chi cùng một ngân sách 1,200 ảnh huấn luyện: giữ
tỉ lệ tự nhiên 2 phần trăm, hay thu có mục tiêu để nhóm con chiếm 10 phần trăm (giả định).

In [ ]:
X_kt_chinh, X_kt_con = X[kt], lech_phai(X[kt])
ket_qua = {}
for ti_le, ten in ((0.02, "tự nhiên, 2 %"), (0.10, "có mục tiêu, 10 %")):
    Xa, ya, ca = tron(hl, ti_le, np.random.default_rng(SEED + 1))
    m = LogisticRegression(max_iter=3000).fit(Xa, ya)
    dung_chinh = m.predict(X_kt_chinh) == y[kt]
    dung_con = m.predict(X_kt_con) == y[kt]
    ket_qua[ten] = (dung_chinh, dung_con)
    print(f"huấn luyện {ten} ({ca.sum()} ảnh nhóm con):")
    do_tren_may("   độ chính xác nhóm chính", 100 * dung_chinh.mean(), "%")
    do_tren_may("   độ chính xác nhóm con", 100 * dung_con.mean(), "%")
    kq, duoi, tren = cong_phat_hanh(int((~dung_con).sum()), len(kt), int((~dung_chinh).sum()), len(kt))
    print(f"   cổng, {len(kt)} ảnh mỗi nhóm: tỉ số sai số trong [{duoi:.2f}, {tren:.2f}] → {kq}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
vi_tri = np.arange(2)
for j, (ten, (dc, dn)) in enumerate(ket_qua.items()):
    gia_tri = [100 * dc.mean(), 100 * dn.mean()]
    sai_so = []
    for d in (dc, dn):
        lo, hi = wilson(int(d.sum()), len(d))
        sai_so.append([100 * d.mean() - 100 * float(lo), 100 * float(hi) - 100 * d.mean()])
    ax.bar(vi_tri + (j - 0.5) * 0.36, gia_tri, width=0.34, color=MAU[j], yerr=np.array(sai_so).T, capsize=4,
           label=f"huấn luyện {ten}")
    for x, v in zip(vi_tri + (j - 0.5) * 0.36, gia_tri):
        ax.text(x, 4, f"{v:.1f} %", ha="center", fontsize=8, color="white")
ax.set_xticks(vi_tri)
ax.set_xticklabels(["nhóm chính", "nhóm con (ảnh lệch 2 điểm ảnh)"])
ax.set_ylim(0, 112)
ax.set_ylabel("độ chính xác (%), kèm khoảng Wilson 95 %")
ax.set_title(f"Cùng {len(hl):,} ảnh huấn luyện, hai cách thu (đo trên máy này)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Với một tập kiểm tra riêng cho nhóm con, khoảng tin cậy hẹp lại đủ để cổng ra quyết định. Còn hình cho
thấy cùng một ngân sách huấn luyện, chia lại theo chủ đích, đổi được độ chính xác của nhóm con nhiều hơn
nhiều so với cái nó lấy đi của nhóm chính. Đây là một mô hình đồ chơi và một nhóm con dựng ra, nhưng
hình dạng của kết quả là điều chương nói: thu thêm dữ liệu theo phân phối tự nhiên gần như không chạm
tới nhóm nhỏ.

## Thử thêm

1. Ở phần 2, đổi `TI_LE_NHOM` thành 0.1 (một nhóm 10 phần trăm) hoặc `BIEN` thành 0.02. Lấy mẫu ngẫu
   nhiên khi ấy cần bao nhiêu ảnh?
2. Ở phần 5, đổi `LOI_MOC` thành 0.10 rồi chạy lại. Khi lỗi của mốc lớn hơn, cần ít hay nhiều mẫu hơn
   để cổng nói "đạt"? Vì sao?
3. Ở phần 6, đổi `buoc` của `lech_phai` thành 1, hoặc `TI_LE_CON` thành 0.05. Khoảng cách giữa hai
   nhóm và kết quả của cổng thay đổi thế nào?

## Tóm lại

* Một con số tổng hợp đổi theo thành phần của bộ kiểm tra, còn sai số của từng nhóm thì không; ở bảng
  2, không thành phần nào làm lộ tỉ số 43.1 lần, và phân tích theo từng yếu tố cũng không.
* Sai số biên xấu nhất 1 điểm phần trăm ở độ tin cậy 95 phần trăm cần khoảng 10,000 mẫu cho nhóm; lấy
  ngẫu nhiên một nhóm 1 phần trăm thì phải thu 1,000,000 mẫu, gấp 100 lần.
* Với 100 mẫu, con số đo của một nhóm dao động rất rộng, và khoảng Wald tự xưng 95 phần trăm phủ thấp
  hơn hẳn khi tỉ lệ lỗi nhỏ; khoảng Wilson thì không.
* Một cổng phát hành so cả khoảng tin cậy với ngưỡng, nên có ba kết quả; chứng minh không có chênh lệch
  cần nhiều mẫu hơn chứng minh có.
* Dữ liệu có chủ đích, cho cả đánh giá lẫn huấn luyện, chạm tới nhóm nhỏ; thêm dữ liệu theo phân phối
  tự nhiên thì gần như không.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật có trách nhiệm — viết lại những gì đặc tả bỏ sót](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch15-responsible-engr/), dựng từ chương
[*Responsible Engineering*](https://mlsysbook.ai/vol1/responsible_engr/responsible_engr.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_15_responsible_engr.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.